# 2. Preprocessing

Validation, encoding, feature engineering and a stratified train/test split for the classification task. The outputs are saved to `data/` and read by notebook 03.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from sklearn.model_selection import train_test_split

from preprocessing import (load_data, validate_data, add_features, encode_categorical,
                           fit_age_cap, apply_age_cap, split_features_target)

df = load_data("../data/students.csv")

## Validation

In [2]:
df = validate_data(df)

[validation] no issues found


## Encoding and feature engineering

Binary variables are already coded 0/1 in the source file. Derived features: pass ratio per semester, grade change between semesters, total enrolled units and whether both parents have higher education. They are created before the categorical grouping and one-hot encoding, which use fixed mappings and no statistics from the data.

In [3]:
df = add_features(df)
df = encode_categorical(df)

print("shape:", df.shape)
print("missing values:", df.isna().sum().sum())
print("students with both parents in higher education:", df["both_parents_higher_ed"].sum())

shape: (4424, 100)
missing values: 0
students with both parents in higher education: 261


## Train/test split

80/20 split, stratified by outcome. The age cap (99th percentile) is estimated on the training set only and then applied to both sets, so no information from the test set is used.

In [4]:
X, y = split_features_target(df, "target")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

age_cap = fit_age_cap(X_train)
X_train, X_test = apply_age_cap(X_train, age_cap), apply_age_cap(X_test, age_cap)

print(f"age cap: {age_cap:.0f}")
print("train:", X_train.shape, "test:", X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

age cap: 50
train: (3539, 99) test: (885, 99)
target
Graduate    0.499
Dropout     0.321
Enrolled    0.179
Name: proportion, dtype: float64


## Save the splits

In [5]:
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)